# TUTORIAL 07: THE FUNDAMENTAL GROUP AND THE STRUCTURE SET

In all previous tutorials, we studied **simply-connected** manifolds (spaces where every loop can be shrunk to a point, meaning $\pi_1(X) = 1$).

But the universe is filled with spaces that have twisted, unshrinkable loops (like a Torus, which has $\pi_1 = \mathbb{Z} \times \mathbb{Z}$). When $\pi_1 \neq 1$, standard homology fails us, and Surgery Theory becomes infinitely more complex.

In this final tutorial, we will explore:
1. **The Fundamental Group ($\pi_1$)**: How to algorithmically extract the group presentation $\langle \text{Generators} \mid \text{Relations} \rangle$ directly from discrete data.
2. **Algebraic K-Theory**: How to compute the Whitehead Group $Wh(\pi_1)$, the absolute obstruction to the s-Cobordism Theorem.
3. **The Structure Set $\mathcal{S}_{TOP}(M)$**: The "Holy Grail" of classification. How to use Sullivan's Formula and the **Surgery Exact Sequence** to find out exactly how many manifolds are homotopy equivalent to $M$ but not homeomorphic.

In [ ]:
from scipy.sparse import csr_matrix
from pysurgery import CWComplex, ChainComplex, extract_pi_1, StructureSet, compute_whitehead_group
print("Advanced Non-Simply-Connected engines loaded.")

## 1. Extracting the Fundamental Group ($\pi_1$)

How do we find the loops in a discrete dataset? We use the **Edge-Path Group** algorithm.

1. We take the 1-skeleton (vertices and edges) of our `CWComplex` and compute a **Maximal Spanning Tree** using Breadth-First Search (BFS).
2. A tree has no loops. Any edge *not* in the tree creates a unique, non-shrinkable loop. These become our **Generators**.
3. But some loops might be filled in by 2-dimensional faces (2-cells). We trace the boundary of every 2-cell to see which generators it wraps around. These become our **Relations**.

In [ ]:
print("--- Example: The Circle S^1 ---")
# 1 Vertex, 1 Edge.
# The edge loops back to the vertex, so its boundary is 0.
c_s1 = CWComplex(
    cells={0: 1, 1: 1},
    attaching_maps={1: csr_matrix((1, 1))}
)

pi1_s1 = extract_pi_1(c_s1)
print(f"Fundamental Group of S^1:\n{pi1_s1}")
print("Explanation: We have 1 generator (the loop) and 0 relations. This is the free group Z!")

In [ ]:
print("\n--- Example: A Filled Disk D^2 ---")
# 1 Vertex, 1 Edge, 1 Face.
# The face is glued directly onto the edge, destroying the loop.
c_d2 = CWComplex(
    cells={0: 1, 1: 1, 2: 1},
    attaching_maps={
        1: csr_matrix((1, 1)), # Edge is a loop
        2: csr_matrix([[1]])   # Face is glued to the edge
    }
)

pi1_d2 = extract_pi_1(c_d2)
print(f"Fundamental Group of D^2:\n{pi1_d2}")
print("Explanation: We have 1 generator, but the relation says 'g_0 = identity'.")
print("The group is trivial. The disk is simply-connected!")

## 2. Algebraic K-Theory & Whitehead Torsion

If two high-dimensional manifolds are homotopy equivalent, are they homeomorphic? 
The **s-Cobordism Theorem** states that they are homeomorphic *if and only if* their Whitehead Torsion vanishes.

The torsion $\tau \in Wh(\pi_1)$, where $Wh(\pi_1)$ is the **Whitehead Group**, a profound object from Algebraic K-Theory defined as $K_1(\mathbb{Z}[\pi_1]) / (\pm \pi_1)$.

`pysurgery` can algorithmically approximate the Whitehead Group based on the $\pi_1$ presentation to inform you if the s-Cobordism theorem is obstructed!

In [ ]:
print("--- Evaluating Whitehead Torsion ---")

wh_d2 = compute_whitehead_group(pi1_d2)
print(f"For the Disk (pi_1 = 1): {wh_d2.description}")

wh_s1 = compute_whitehead_group(pi1_s1)
print(f"For the Circle (pi_1 = Z): {wh_s1.description}")

print("\nThis computationally confirms the famous Bass-Heller-Swan theorem that Wh(Z) = 0, meaning an s-cobordism over a circle is always a trivial product cylinder!")

## 3. The Structure Set and Normal Invariants

In high dimensions ($n \ge 5$), the ultimate goal of topology is to compute the **Structure Set** $\mathcal{S}_{TOP}(M)$. 
This set contains all the distinct manifolds that are homotopy equivalent to $M$ (they look the same if you squint), but are topologically distinct (not homeomorphic).

To find this set, we use the **Surgery Exact Sequence**:
$$ \dots \to L_{n+1}(\pi_1) \to \mathcal{S}_{TOP}(M) \to [M, G/TOP] \to L_n(\pi_1) $$

- **$[M, G/TOP]$**: The set of **Normal Invariants**. These are essentially different vector bundles you can put on $M$ to attempt surgery. `pysurgery` computes the rank of this set using **Sullivan's Characteristic Variety Formula**.
- **$L_n(\pi_1)$**: The **Wall Group**. This is the obstruction. If a normal invariant maps to a non-zero element in $L_n$, it means the surgery *fails*.
- **$L_{n+1}(\pi_1)$**: This group acts on the Structure Set. It dictates how many different smooth/topological structures can exist on the *same* manifold (e.g., Exotic Spheres!).

In [ ]:
print("--- Evaluating Normal Invariants for a 5D Sphere ---")
# S^5 has H_0=Z, H_5=Z. All other homologies are 0.
c5 = ChainComplex(boundaries={i: csr_matrix((1,0)) for i in range(1,6)}, dimensions=list(range(6)))

s_set = StructureSet(dimension=5, fundamental_group="1")

normal_invariants_report = s_set.compute_normal_invariants(c5)
print(normal_invariants_report)

print("\n--- Evaluating the Full Surgery Exact Sequence ---")
report = s_set.evaluate_exact_sequence()
print(report)

### Mathematical Insight
Look at the report above! 

For a 5D manifold, Sullivan's formula sums over $H^4(M; \mathbb{Z})$ and $H^2(M; \mathbb{Z}_2)$. Because this is $S^5$, those groups are 0, so the set of Normal Invariants $[M, G/TOP]$ has Rank 0.

The sequence ends in $L_5(1)$. 
Because $5 \equiv 1 \pmod 4$, Wall's classification tells us that $L_5(1) = 0$.

What does this mean? It means **the surgery obstruction completely vanishes**. 

However, the sequence starts with $L_6(1)$, which is $Z_2$ (since $6 \equiv 2 \pmod 4$, related to the Arf Invariant). This $Z_2$ acts on the Structure Set, implying there is a deep, hidden $\mathbb{Z}_2$ ambiguity in the classification of 5-manifolds.

## Final Conclusion
You have now reached the absolute pinnacle of `pysurgery`.
You have moved from discrete vertices and matrices all the way to extracting non-abelian fundamental group presentations, resolving the s-Cobordism theorem via K-Theory, and calculating Normal Invariants and L-Groups to execute the High-Dimensional Surgery Exact Sequence!